## Importing Packages

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from xgboost import XGBRegressor

import mlflow
import mlflow.sklearn
import mlflow.xgboost

## Load the Data

In [2]:
data = fetch_california_housing(as_frame=True)

X = data.data
y = data.target

In [3]:
print("Shape:", X.shape)
print("Target:", data.target_names)
print(y.describe())

Shape: (20640, 8)
Target: ['MedHouseVal']
count    20640.000000
mean         2.068558
std          1.153956
min          0.149990
25%          1.196000
50%          1.797000
75%          2.647250
max          5.000010
Name: MedHouseVal, dtype: float64


## Train Ready

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## Diff Exp Models

In [5]:
lr = LinearRegression()

lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

print("R2:", r2_score(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))

R2: 0.5757877060324514
MSE: 0.5558915986952435


In [6]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print("R2:", r2_score(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))

R2: 0.774774131273305
MSE: 0.29513800051156747


In [7]:
xgb = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

xgb.fit(X_train, y_train)

y_pred = xgb.predict(X_test)

print("R2:", r2_score(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))

R2: 0.8408716044998452
MSE: 0.2085232781564716


## Prepare for ML Flow

In [8]:
models = [

    (
        "Linear Regression",
        LinearRegression(),
        (X_train, y_train),
        (X_test, y_test)
    ),

    (
        "Random Forest",
        RandomForestRegressor(
            n_estimators=200,
            max_depth=10,
            random_state=42
        ),
        (X_train, y_train),
        (X_test, y_test)
    ),

    (
        "XGBoost",
        XGBRegressor(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            random_state=42
        ),
        (X_train, y_train),
        (X_test, y_test)
    )

]

In [9]:
reports = []

for model_name, model, train_set, test_set in models:

    X_tr, y_tr = train_set
    X_te, y_te = test_set

    model.fit(X_tr, y_tr)

    predictions = model.predict(X_te)

    report = {
        "mse": mean_squared_error(y_te, predictions),
        "rmse": float(np.sqrt(mean_squared_error(y_te, predictions))),
        "mae": mean_absolute_error(y_te, predictions),
        "r2": r2_score(y_te, predictions)
    }

    reports.append(report)

In [10]:
reports

[{'mse': 0.5558915986952435,
  'rmse': 0.7455813830127758,
  'mae': 0.5332001304956557,
  'r2': 0.5757877060324514},
 {'mse': 0.29513800051156747,
  'rmse': 0.5432660494744426,
  'mae': 0.3657301973669837,
  'r2': 0.774774131273305},
 {'mse': 0.2085232781564716,
  'rmse': 0.45664349131075066,
  'mae': 0.2959191474993146,
  'r2': 0.8408716044998452}]